In [49]:
import pyodbc
import pandas as pd
import numpy as np

In [50]:
conn_172 = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=172.22.24.232;'  # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;'         # Tên cơ sở dữ liệu
    'UID=sa;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)

In [51]:
query_phieumuon = """SELECT TOP (100) PMS.ID_phieu_muon, 
				                    PMS.ID_ban_doc, 
				                    PMS.ID_tai_lieu, 
				                    Ma_xep_gia, 
				                    Ngay_muon,
				                    Ngay_tra,
				                    So_luot_gia_han,
				                    So_ngay_qua_han
                    FROM oltp.Phieu_muon_sach PMS
                        JOIN olap.DIM_Ban_doc BD ON BD.ID_ban_doc = PMS.ID_ban_doc
                        JOIN olap.DIM_Xep_gia XG ON XG.ID_xep_gia = PMS.ID_xep_gia
                    WHERE So_luot_gia_han IS NOT NULL AND So_luot_gia_han != 0 AND Ngay_tra = 0"""
df_phieumuon = pd.read_sql(query_phieumuon, conn_172)
print(df_phieumuon)

    ID_phieu_muon ID_ban_doc  ID_tai_lieu Ma_xep_gia  Ngay_muon  Ngay_tra  \
0          740360   05118009         7065  SKV022321   20070910         0   
1          960367   07709021        11227  SKV026359   20080401         0   
2          960368   07709021        17303  SKV044363   20080401         0   
3          405144   04105105        12200  SKV036633   20060503         0   
4           16783   01709026          519  SKV001889   20021206         0   
5          193740   03705011        11062  SKV026143   20041026         0   
6          206314   03101372          576  SKV002831   20041206         0   
7          199472   02104022         1875  SKV010162   20041115         0   
8          200664   02122037         2703  SKV014000   20041118         0   
9          174454   02113012          674  SKV003767   20040916         0   
10         740361   05118009        19424  SKV052281   20070910         0   
11        1150972   06108099        18260  SKV059254   20081029         0   

C:\Users\phung\AppData\Local\Temp\ipykernel_14124\1099599717.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_phieumuon = pd.read_sql(query_phieumuon, conn_172)


In [52]:
# So_luot_muon
so_luot_muon = df_phieumuon.groupby(['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'])['ID_phieu_muon'].count().reset_index()
so_luot_muon = so_luot_muon.rename(columns={'ID_phieu_muon': 'So_luot_muon'})
print(so_luot_muon)

   ID_ban_doc  ID_tai_lieu Ma_xep_gia  Ngay_muon  So_luot_muon
0    01709026          519  SKV001889   20021206             1
1    02104022         1875  SKV010162   20041115             1
2    02113012          674  SKV003767   20040916             1
3    02122037         2703  SKV014000   20041118             1
4    03101372          576  SKV002831   20041206             1
5    03103154         2009  SKV010718   20071001             1
6    03103154         2011  SKV010722   20071001             1
7    03103154        19774  SKV055880   20071001             1
8    03705011        11062  SKV026143   20041026             1
9    04105105        12200  SKV036633   20060503             1
10   05118009         7065  SKV022321   20070910             1
11   05118009        19424  SKV052281   20070910             1
12   06108099        18260  SKV059254   20081029             1
13   07709021        11227  SKV026359   20080401             1
14   07709021        17303  SKV044363   20080401       

In [53]:
# So_luot_da_hoan_tra: Ngay_tra != 0
df_phieumuon_da_tra = df_phieumuon[df_phieumuon['Ngay_tra'] != 0]
so_luot_da_tra = df_phieumuon_da_tra.groupby(['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'])['ID_phieu_muon'].count().reset_index()
so_luot_da_tra = so_luot_da_tra.rename(columns={'ID_phieu_muon': 'So_luot_da_hoan_tra'})

print(so_luot_da_tra)

Empty DataFrame
Columns: [ID_ban_doc, ID_tai_lieu, Ma_xep_gia, Ngay_muon, So_luot_da_hoan_tra]
Index: []


In [54]:
# So_luot_khong_hoan_tra: Ngay_tra == 0
df_phieumuon_khong_tra = df_phieumuon[df_phieumuon['Ngay_tra'] == 0]
so_luot_khong_tra = df_phieumuon_khong_tra.groupby(['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'])['ID_phieu_muon'].count().reset_index()
so_luot_khong_tra = so_luot_khong_tra.rename(columns={'ID_phieu_muon': 'So_luot_khong_hoan_tra'})
print(so_luot_khong_tra)

   ID_ban_doc  ID_tai_lieu Ma_xep_gia  Ngay_muon  So_luot_khong_hoan_tra
0    01709026          519  SKV001889   20021206                       1
1    02104022         1875  SKV010162   20041115                       1
2    02113012          674  SKV003767   20040916                       1
3    02122037         2703  SKV014000   20041118                       1
4    03101372          576  SKV002831   20041206                       1
5    03103154         2009  SKV010718   20071001                       1
6    03103154         2011  SKV010722   20071001                       1
7    03103154        19774  SKV055880   20071001                       1
8    03705011        11062  SKV026143   20041026                       1
9    04105105        12200  SKV036633   20060503                       1
10   05118009         7065  SKV022321   20070910                       1
11   05118009        19424  SKV052281   20070910                       1
12   06108099        18260  SKV059254   20081029   

In [55]:
# So_luot_qua_han: So_luot_gia_han not null
df_phieumuon_qua_han = df_phieumuon[df_phieumuon['So_luot_gia_han'].notnull()]
so_luot_qua_han = df_phieumuon_qua_han.groupby(['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'])['ID_phieu_muon'].count().reset_index()
so_luot_qua_han = so_luot_qua_han.rename(columns={'ID_phieu_muon': 'So_luot_qua_han'})
print(so_luot_qua_han)

   ID_ban_doc  ID_tai_lieu Ma_xep_gia  Ngay_muon  So_luot_qua_han
0    01709026          519  SKV001889   20021206                1
1    02104022         1875  SKV010162   20041115                1
2    02113012          674  SKV003767   20040916                1
3    02122037         2703  SKV014000   20041118                1
4    03101372          576  SKV002831   20041206                1
5    03103154         2009  SKV010718   20071001                1
6    03103154         2011  SKV010722   20071001                1
7    03103154        19774  SKV055880   20071001                1
8    03705011        11062  SKV026143   20041026                1
9    04105105        12200  SKV036633   20060503                1
10   05118009         7065  SKV022321   20070910                1
11   05118009        19424  SKV052281   20070910                1
12   06108099        18260  SKV059254   20081029                1
13   07709021        11227  SKV026359   20080401                1
14   07709

In [56]:
# Gộp tất cả lại
df_final = so_luot_muon \
    .merge(so_luot_da_tra, on=['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'], how='left') \
    .merge(so_luot_khong_tra, on=['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'], how='left') \
    .merge(so_luot_qua_han, on=['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'], how='left')

# Fill các NaN bằng 0
df_final = df_final.fillna(0).astype({
    'So_luot_da_hoan_tra': int,
    'So_luot_khong_hoan_tra': int,
    'So_luot_qua_han': int
})

# ✅ In thử kết quả
print(df_final.head(10))

  ID_ban_doc  ID_tai_lieu Ma_xep_gia  Ngay_muon  So_luot_muon  \
0   01709026          519  SKV001889   20021206             1   
1   02104022         1875  SKV010162   20041115             1   
2   02113012          674  SKV003767   20040916             1   
3   02122037         2703  SKV014000   20041118             1   
4   03101372          576  SKV002831   20041206             1   
5   03103154         2009  SKV010718   20071001             1   
6   03103154         2011  SKV010722   20071001             1   
7   03103154        19774  SKV055880   20071001             1   
8   03705011        11062  SKV026143   20041026             1   
9   04105105        12200  SKV036633   20060503             1   

   So_luot_da_hoan_tra  So_luot_khong_hoan_tra  So_luot_qua_han  
0                    0                       1                1  
1                    0                       1                1  
2                    0                       1                1  
3                   

In [ ]:
# Load vào datawarehouse
cursor_dwh = conn_172.cursor()

insert_query = """
    INSERT INTO olap.FACT_Muon (
        ID_ban_doc,
        ID_tai_lieu, 
        Ma_xep_gia, 
        ID_date, 
        So_luot_muon,
        So_luot_da_hoan_tra,
        So_luot_khong_hoan_tra,
        So_luot_qua_han
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

for index, row in df_final.iterrows():
    values = (
        row['ID_ban_doc'],
        row['ID_tai_lieu'],
        row['Ma_xep_gia'],
        row['Ngay_muon'],  # Đây là ID_date
        int(row['So_luot_muon']),
        int(row.get('So_luot_da_hoan_tra', 0)),
        int(row.get('So_luot_khong_hoan_tra', 0)),
        int(row.get('So_luot_qua_han', 0))
    )
    cursor_dwh.execute(insert_query, values)

conn_172.commit()
